# 验证集三模型多数票

先比较此前讨论过的 B06、TextCNN、单向 LSTM，再比较 B06、TextCNN、DistilBERT。每个模型投一张类别票，至少两票判为正类时输出正类。沿用已固定的检查点、B06 训练流程和相同验证集。此处是探索性对照，包含两个固定组合及其对应的单模型参照。


In [ ]:
import hashlib
import json
import math

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.svm import LinearSVC
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from sentiment_analysis.evaluation import classification_metrics
from sentiment_analysis.lstm import LSTMClassifier
from sentiment_analysis.paths import ROOT, CONFIGS_DIR, DATA_DIR, TABLES_DIR
from sentiment_analysis.sequence_data import SentimentDataset, TOKEN_PATTERN
from sentiment_analysis.textcnn import TextCNN

def sha256(path):
    with path.open("rb") as file:
        return hashlib.file_digest(file, "sha256").hexdigest()

protocol = json.loads((CONFIGS_DIR / "final_test_protocol.json").read_text(encoding="utf-8"))
for name in ["train", "validation", "tfidf", "textcnn", "lstm"]:
    record = protocol["files"][name]
    if sha256(ROOT / record["path"]) != record["sha256"]:
        raise ValueError(f"Fixed input changed: {name}")


In [ ]:
# 恢复 B06 并核对验证结果
train_data = pd.read_csv(ROOT / protocol["files"]["train"]["path"])
validation_data = pd.read_csv(ROOT / protocol["files"]["validation"]["path"])
vectorizer = joblib.load(ROOT / protocol["files"]["tfidf"]["path"])
X_train = vectorizer.transform(train_data["text"])
X_validation = vectorizer.transform(validation_data["text"])
svm_model = LinearSVC(C=1.0, max_iter=1000, random_state=42)
svm_model.fit(X_train, train_data["label"])
validation_prediction = svm_model.predict(X_validation)
validation_metrics = classification_metrics(validation_data["label"], validation_prediction)
baseline_record = pd.read_csv(ROOT / protocol["files"]["baseline_validation"]["path"])
baseline_record = baseline_record[baseline_record["EXPERIMENT"] == "B06"].iloc[0]
for metric, value in validation_metrics.items():
    if not np.isclose(value, baseline_record[metric], rtol=0, atol=1e-12):
        raise ValueError(f"B06 validation metric does not match: {metric}")
print("B06 validation results verified.")


In [ ]:
# 生成 CNN 和单向 LSTM 在同一验证集上的原有类别判定
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cnn = torch.load(ROOT / protocol["files"]["textcnn"]["path"], map_location="cpu", weights_only=True)
lstm = torch.load(ROOT / protocol["files"]["lstm"]["path"], map_location="cpu", weights_only=True)
if (cnn["vocabulary"] != lstm["vocabulary"] or cnn["max_length"] != lstm["max_length"]
        or any(item["token_pattern"] != TOKEN_PATTERN.pattern or not item["lowercase"] for item in [cnn, lstm])):
    raise ValueError("The checkpoints disagree on sequence processing.")
models = {
    "TextCNN": TextCNN(**cnn["model_config"]).to(device),
    "LSTM": LSTMClassifier(**lstm["model_config"]).to(device),
}
for name, checkpoint in [("TextCNN", cnn), ("LSTM", lstm)]:
    models[name].load_state_dict(checkpoint["model_state_dict"])
    models[name].eval()
dataset = SentimentDataset(validation_data, cnn["vocabulary"], cnn["max_length"])
loader = DataLoader(dataset, batch_size=32, shuffle=False)
labels = validation_data["label"].to_numpy()
all_predictions = {"B06": validation_prediction, "TextCNN": [], "LSTM": []}
with torch.inference_mode():
    offset = 0
    for token_ids, lengths, batch_labels in loader:
        end = offset + len(batch_labels)
        if not np.array_equal(batch_labels.numpy(), labels[offset:end]):
            raise ValueError("Neural model labels do not align with validation rows.")
        for name, model in models.items():
            logits = model(token_ids.to(device), lengths.to(device))
            all_predictions[name].extend((logits.softmax(dim=1)[:, 1] > 0.5).long().cpu().tolist())
        offset = end
if offset != len(labels):
    raise ValueError("Missing neural model predictions.")


In [ ]:
# 使用 DistilBERT 固定检查点和原评估配置
model_dir = DATA_DIR / "models" / "distilbert_seed42"
protocol = json.loads((CONFIGS_DIR / "distilbert_test_protocol.json").read_text(encoding="utf-8"))
training_path = CONFIGS_DIR / "distilbert_finetuning.json"
training = json.loads(training_path.read_text(encoding="utf-8"))
if (sha256(training_path) != protocol["training_protocol_sha256"]
        or sha256(ROOT / "data/cleaned/val.csv") != training["split_sha256"]["validation"]):
    raise ValueError("DistilBERT validation data or training protocol changed.")
for name, digest in protocol["checkpoint_sha256"].items():
    if sha256(model_dir / name) != digest:
        raise ValueError(f"DistilBERT checkpoint changed: {name}")
if device.type != "cuda" or not torch.cuda.is_bf16_supported():
    raise RuntimeError("The fixed DistilBERT evaluation requires CUDA BF16.")
# 先恢复已选检查点并核对验证指标
tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True, trust_remote_code=False)
model = AutoModelForSequenceClassification.from_pretrained(
    model_dir, local_files_only=True, use_safetensors=True, trust_remote_code=False,
    attn_implementation=protocol["attention_implementation"],
).to(device)
model.eval()
if model.config.id2label != {0: "NEGATIVE", 1: "POSITIVE"}:
    raise ValueError("Checkpoint label mapping does not match the evaluation.")

def predict(data, name):
    labels = []
    predictions = []
    probabilities = []
    loss_sum = 0.0
    truncated = 0
    batch_size = protocol["batch_size"]
    with torch.inference_mode():
        for start in range(0, len(data), batch_size):
            rows = data.iloc[start:start + batch_size]
            texts = rows["text"].tolist()
            lengths = tokenizer(texts, truncation=False, return_length=True, verbose=False)["length"]
            truncated += sum(length > protocol["max_length"] for length in lengths)
            batch = tokenizer(
                texts, truncation=True, max_length=protocol["max_length"], padding=True,
                pad_to_multiple_of=protocol["pad_to_multiple_of"], return_tensors="pt",
                return_token_type_ids=False,
            ).to(device)
            batch_labels = torch.tensor(rows["label"].tolist(), dtype=torch.long, device=device)
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                output = model(**batch, labels=batch_labels)
            loss_sum += float(output.loss) * len(rows)
            labels.extend(batch_labels.cpu().tolist())
            predictions.extend(output.logits.argmax(dim=1).cpu().tolist())
            probabilities.extend(output.logits.float().softmax(dim=1)[:, 1].cpu().tolist())
            if start and start % (batch_size * 500) == 0:
                print(f"{name}: {start:,}/{len(data):,}", flush=True)
    if labels != data["label"].tolist() or not math.isfinite(loss_sum):
        raise ValueError(f"{name} labels or loss are invalid.")
    return loss_sum / len(labels), predictions, probabilities, truncated


_, distilbert_predictions, _, _ = predict(validation_data, "Validation")
all_predictions["DistilBERT"] = distilbert_predictions


In [ ]:
# 复核单模型指标，计算多数票和逐样本纠错变化
reference = pd.read_csv(TABLES_DIR / "distilbert_validation_comparison.csv").set_index("MODEL")
rows = []
for name, prediction in all_predictions.items():
    prediction = np.asarray(prediction)
    if prediction.shape != labels.shape:
        raise ValueError(f"Invalid prediction shape for {name}.")
    metrics = classification_metrics(labels, prediction)
    for metric, value in metrics.items():
        if not math.isclose(value, reference.loc[name, "VALIDATION " + metric], abs_tol=1e-12):
            raise ValueError(f"Original validation metric did not reproduce: {name} {metric}")
    rows.append({"MODEL": name, **metrics})
    all_predictions[name] = prediction

comparisons = [
    (("B06", "TextCNN", "LSTM"), "B06"),
    (("B06", "TextCNN", "DistilBERT"), "DistilBERT"),
]
changes = []
for members, anchor in comparisons:
    name = " + ".join(members) + " majority"
    prediction = (sum(all_predictions[member] for member in members) >= 2).astype(int)
    anchor_correct = all_predictions[anchor] == labels
    vote_correct = prediction == labels
    rows.append({"MODEL": name, **classification_metrics(labels, prediction)})
    changes.append({
        "MODEL": name, "REFERENCE": anchor,
        "CORRECTED": int((~anchor_correct & vote_correct).sum()),
        "NEW ERRORS": int((anchor_correct & ~vote_correct).sum()),
    })

results = pd.DataFrame(rows).merge(pd.DataFrame(changes), on="MODEL", how="left")
for name in ["CORRECTED", "NEW ERRORS"]:
    results[name] = results[name].astype("Int64")
results.to_csv(TABLES_DIR / "three_model_voting_validation.csv", index=False)
display(results.round(4))


## 结果

B06 + TextCNN + 单向 LSTM 的验证 F1 为 0.8939，相对 B06 的 0.8889，纠正 79 条并引入 62 条错误。B06 + TextCNN + DistilBERT 的验证 F1 为 0.9179，相对 DistilBERT 的 0.9088，纠正 89 条并引入 65 条错误。

两组多数票在本次验证集上都有小幅净收益。此前验证集已用于模型选择，因此这两组结果是探索性观察。历史测试结果对应此前的固定方案。
